<!-- nav -->
[← 7 · One rule, many languages](07_One_Rule_Many_Languages.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 8 · Rules for firmware and hardware

**The problem.** The logger's firmware, in C, checks each packet before sending it, and its FPGA's verification
engineers check the same condition with SystemVerilog assertions. In C, `uint8_t + uint8_t` is an `int`, and signed
overflow is undefined. In SystemVerilog, a bit can be `x`, unknown, or `z`, undriven. A rule that is right in Python
can be wrong in either.

The Ccpp and SystemVerilog dialects model those languages' rules exactly, so you can translate the packet rule of case
study 4 and evaluate it the way the firmware and the hardware will.

In [1]:
from mbse.Expressions import Domains as D, Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects import Ccpp, SystemVerilog
from toolkit import show

int16 = D.OfInteger.Builder().width(16).create()
uint8 = D.OfInteger.Builder().width(8).signed(False).create()
r, s = E.variable("raw"), E.variable("status")
packet_ok = (r.ge(E.literal(200, int16)).and_(r.le(E.literal(800, int16)))
             .and_(s.bitand(E.literal(0b100, uint8)).eq(E.literal(0, uint8)))).data
to_c = Translators.between(E.DIALECT, Ccpp.Expressions.DIALECT)
to_sv = Translators.between(E.DIALECT, SystemVerilog.Expressions.DIALECT)

## Step 1: C

The translation writes typed literals as C writes them, with casts to the `<stdint.h>` types:

In [2]:
c = to_c.forward(packet_ok)
print(Ccpp.Text.ToText(c))
for status in (0, 0b100):
    print(status, Ccpp.Evaluators.OfAny(c, {"raw": D.Value(int16, 523), "status": D.Value(uint8, status)}))

raw >= (int16_t)200 && raw <= (int16_t)800 && (status & (uint8_t)4) == (uint8_t)0
0 True
4 False


## Step 2: where C differs

C promotes small integers to `int` before arithmetic. Basic has no implied promotions (case study 4), so the same sum
overflows a `uint8` in Basic and is simply 300 in C:

In [3]:
total = E.literal(200, uint8).add(E.literal(100, uint8)).data
print(Ccpp.Text.ToText(to_c.forward(total)), "=", Ccpp.Evaluators.OfAny(to_c.forward(total), {}))
try:
    Evaluators.OfAny(total)
except OverflowError as error:
    print("Basic:", error)

(uint8_t)200 + (uint8_t)100 = 300
Basic: add overflows uint8: 300


Signed overflow is undefined behavior in C: a compiler may assume it never happens. The evaluator raises instead of
picking a result, which is what a test should do:

In [4]:
try:
    Ccpp.Evaluators.OfAny(Ccpp.Expressions.binary("+", Ccpp.Expressions.identifier("x"), 1), {"x": 2147483647})
except OverflowError as error:
    print(error)

signed overflow in +: 2147483648 does not fit 'int'


## Step 3: SystemVerilog

SystemVerilog writes sized literals: `16'sd200` is a signed 16-bit decimal. Its evaluator gives a 4-state value,
written as SystemVerilog would display it:

In [5]:
sv = to_sv.forward(packet_ok)
print(SystemVerilog.Text.ToText(sv))
SystemVerilog.Evaluators.OfAny(sv, {"raw": D.Value(int16, 523), "status": D.Value(uint8, 0)})

raw >= 16'sd200 && raw <= 16'sd800 && (status & 8'd4) == 8'd0


1'b1

## Step 4: x and z

In simulation, a register that was never reset holds `x`. Whether that matters depends on the bit: the mask
looks only at bit 2, so an `x` elsewhere doesn't change the answer, but an `x` in bit 2 makes the whole condition `x`,
which an assertion reports as a failure to know. `Logic.of` makes a 4-state value from its bits:

In [6]:
Logic = SystemVerilog.Domains.Logic
for bits in ("00000000", "00000100", "000000x0", "00000x00"):
    print(bits, SystemVerilog.Evaluators.OfAny(sv, {"raw": D.Value(int16, 523), "status": Logic.of(bits)}))

00000000 1'b1
00000100 1'b0
000000x0 1'b1
00000x00 1'bx


This is Kleene's unknown again, in hardware's terms: `x` where the data doesn't decide.

## Step 5: what has no counterpart

`has` asks whether a property is present. A C struct or a SystemVerilog register always has its fields, so there's
nothing to translate it to, and the translator refuses, naming the term:

In [7]:
for translator in (to_c, to_sv):
    try:
        translator.forward(E.variable("this").has("celsius"))
    except ValueError as error:
        print(error)

Basic operation 'has' has no Ccpp counterpart
Basic operation 'has' has no SystemVerilog counterpart


## Why it works this way

- **Model the language, don't approximate it.** Firmware and hardware bugs live in exactly the corners where languages
  differ: promotions, wrapping, undefined behavior, `x`. An evaluator that follows C's or IEEE 1800's rules finds them
  in a test, before the hardware does.
- **The neutral rule stays neutral.** Basic doesn't take C's promotions or SystemVerilog's sizing. The translator
  writes them out, and the target's evaluator applies them.
- **Refuse what can't be said.** A rule about whether data is present has no meaning for a fixed-layout packet; a
  silent approximation would be a bug in the translation.

That's the series: a rule written once, as data, checked, stored, analyzed, specialized, and run in each language its
users work in.

<!-- nav -->
[← 7 · One rule, many languages](07_One_Rule_Many_Languages.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)